In [1]:
import onnxruntime as ort
print(ort.get_available_providers())
# Debe incluir: 'CUDAExecutionProvider'


['AzureExecutionProvider', 'CPUExecutionProvider']


In [2]:
#VIDEO = "4640663MA1.mp4"   # <- cambia esto si tu archivo se llama distinto


In [ ]:
# ===== FaceIDs con InsightFace + ONNX Runtime (GPU si disponible) ============
import onnxruntime as ort
providers = ort.get_available_providers()
print("ONNX Runtime providers:", providers)

use_gpu = "CUDAExecutionProvider" in providers

# Dependencias
import cv2, numpy as np
from pathlib import Path
from sklearn.cluster import DBSCAN
from tqdm import tqdm

# InsightFace
from insightface.app import FaceAnalysis

VIDEO = "4640663MA1.mp4"    # <-- cambia por tu vídeo
SAMPLE_FPS = 5         # procesa ~5 fps
GAP_MERGE_S = 1.0      # fusión de huecos
DBSCAN_EPS = 0.45
MIN_SAMPLES = 3

# Preparar InsightFace (sin 'providers' porque tu versión no lo acepta)
app = FaceAnalysis(name="buffalo_l")
if use_gpu:
    app.prepare(ctx_id=0, det_size=(640, 640))   # GPU
else:
    app.prepare(ctx_id=-1, det_size=(640, 640))  # CPU

# Cargar vídeo
cap = cv2.VideoCapture(VIDEO)
if not cap.isOpened():
    raise SystemExit(f"No se pudo abrir el vídeo: {VIDEO}")
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
step = max(1, int(round(fps / SAMPLE_FPS)))

embeds, times, bboxes = [], [], []
f = 0
ok, frame = cap.read()
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
with tqdm(total=total, desc="Procesando vídeo") as pbar:
    while ok:
        if f % step == 0:
            faces = app.get(frame)
            t = f / fps
            for fa in faces:
                if fa.normed_embedding is None:
                    continue
                embeds.append(fa.normed_embedding)
                times.append(t)
                bboxes.append(fa.bbox.astype(int).tolist())
        ok, frame = cap.read(); f += 1; pbar.update(1)
cap.release()

if not embeds:
    raise SystemExit("No se detectaron caras. Sube SAMPLE_FPS, cambia det_size o revisa el vídeo.")

# Clustering por similitud (cosine)
X = np.vstack(embeds)
cl = DBSCAN(eps=DBSCAN_EPS, min_samples=MIN_SAMPLES, metric="cosine").fit(X)
labels = cl.labels_           # -1 = ruido
valid = labels >= 0
face_ids = sorted(set(labels[valid]))
print("FaceIDs detectados:", face_ids)

# Construir segmentos por FaceID
from collections import defaultdict
per_id = defaultdict(list)
for t, lab in zip(times, labels):
    if lab >= 0:
        per_id[int(lab)].append(t)

rows = []
for fid, ts in per_id.items():
    ts = sorted(ts)
    start = prev = ts[0]
    for t in ts[1:]:
        if t - prev > GAP_MERGE_S:
            rows.append((fid, start, prev))
            start = t
        prev = t
    rows.append((fid, start, prev))

# Guardar CSV
out_segments = Path("faces_segments.csv")
with out_segments.open("w", encoding="utf-8") as f:
    f.write("face_id,start_s,end_s\n")
    for fid, s, e in rows:
        f.write(f"{fid},{s:.3f},{e:.3f}\n")

print("Segmentos por FaceID ->", out_segments.resolve())
# ============================================================================


ONNX Runtime providers: ['AzureExecutionProvider', 'CPUExecutionProvider']
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\carlos.basallote/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\carlos.basallote/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\carlos.basallote/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\carlos.basallote/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvid

Procesando vídeo:   3%|▎         | 686/24192 [00:07<04:09, 94.27it/s] c:\Users\carlos.basallote\.conda\envs\torch-cu124\Lib\site-packages\insightface\utils\transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4
Procesando vídeo:  73%|███████▎  | 17651/24192 [50:40<15:36,  6.98it/s]  